# Map-Reduce with Send API

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will learn how to use `Send` objects from conditional edges to create dynamic parallel workers, aggregate results with list reducers, and build the orchestrator-worker pattern.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic Map-Reduce with Send

Create parallel workers dynamically using `Send` objects from a conditional edge.

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class OverallState(TypedDict):
    topics: list[str]
    results: Annotated[list[str], operator.add]

class WorkerState(TypedDict):
    topic: str

def orchestrator(state: OverallState) -> dict:
    return {"topics": state["topics"]}

def route_work(state: OverallState) -> list[Send]:
    return [Send("worker", {"topic": t}) for t in state["topics"]]

def worker(state: WorkerState) -> dict:
    result = f"Report on: {state['topic']}"
    return {"results": [result]}

graph = StateGraph(OverallState)
graph.add_node("orchestrator", orchestrator)
graph.add_node("worker", worker)
graph.add_conditional_edges("orchestrator", route_work, path_map=["worker"])
graph.add_edge(START, "orchestrator")
graph.add_edge("worker", END)

app = graph.compile()

result = app.invoke({"topics": ["AI", "Quantum Computing", "Robotics"]})
print(f"Number of results: {len(result['results'])}")
for r in result["results"]:
    print(f"  - {r}")

## 4. Dynamic Worker Count with LLM

The number of workers is determined at runtime by the LLM's structured output.

In [ ]:
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

class Section(BaseModel):
    title: str
    description: str

class Plan(BaseModel):
    sections: list[Section]

class ArticleState(TypedDict):
    topic: str
    sections: Annotated[list[str], operator.add]

class SectionState(TypedDict):
    title: str
    description: str

llm = ChatOpenAI(model="gpt-4o-mini")

def planner(state: ArticleState) -> dict:
    return {"sections": []}

def route_sections(state: ArticleState) -> list[Send]:
    plan = llm.with_structured_output(Plan).invoke(
        f"Create 3-5 section titles for an article about: {state['topic']}"
    )
    return [
        Send("section_writer", {"title": s.title, "description": s.description})
        for s in plan.sections
    ]

def section_writer(state: SectionState) -> dict:
    content = llm.invoke(
        f"Write one paragraph about: {state['title']} - {state['description']}"
    )
    return {"sections": [f"## {state['title']}\n{content.content}"]}

graph2 = StateGraph(ArticleState)
graph2.add_node("planner", planner)
graph2.add_node("section_writer", section_writer)
graph2.add_conditional_edges("planner", route_sections, path_map=["section_writer"])
graph2.add_edge(START, "planner")
graph2.add_edge("section_writer", END)

app2 = graph2.compile()

result = app2.invoke({"topic": "Machine Learning in Healthcare"})
print(f"Sections generated: {len(result['sections'])}")
for section in result["sections"]:
    print(f"\n{section[:100]}...")

## 5. Orchestrator-Worker Pattern

The full pattern: plan → parallel execute → synthesize.

In [ ]:
class OrchestratorState(TypedDict):
    query: str
    tasks: list[str]
    results: Annotated[list[str], operator.add]
    final_answer: str

class TaskState(TypedDict):
    task: str

def plan(state: OrchestratorState) -> dict:
    tasks = [
        f"Research: {state['query']}",
        f"Find examples: {state['query']}",
        f"Check alternatives: {state['query']}",
    ]
    return {"tasks": tasks}

def distribute(state: OrchestratorState) -> list[Send]:
    return [Send("execute", {"task": t}) for t in state["tasks"]]

def execute(state: TaskState) -> dict:
    response = llm.invoke(f"Complete this task concisely: {state['task']}")
    return {"results": [f"{state['task']}:\n{response.content}"]}

def synthesize(state: OrchestratorState) -> dict:
    combined = "\n\n".join(state["results"])
    response = llm.invoke(
        f"Synthesize these results into a final answer:\n\n{combined}"
    )
    return {"final_answer": response.content}

graph3 = StateGraph(OrchestratorState)
graph3.add_node("plan", plan)
graph3.add_node("execute", execute)
graph3.add_node("synthesize", synthesize)
graph3.add_conditional_edges("plan", distribute, path_map=["execute"])
graph3.add_edge(START, "plan")
graph3.add_edge("execute", "synthesize")
graph3.add_edge("synthesize", END)

app3 = graph3.compile()

result = app3.invoke({"query": "best practices for LangGraph deployment"})
print(f"Tasks completed: {len(result['results'])}")
print(f"\nFinal Answer:\n{result['final_answer'][:300]}...")

## 6. Verify Aggregation

Confirm that `operator.add` correctly aggregates results from all parallel workers.

In [ ]:
class AggState(TypedDict):
    items: list[str]
    outputs: Annotated[list[str], operator.add]

class ItemState(TypedDict):
    item: str

def start_agg(state: AggState) -> dict:
    return {}

def fan_out(state: AggState) -> list[Send]:
    return [Send("process", {"item": i}) for i in state["items"]]

def process(state: ItemState) -> dict:
    return {"outputs": [f"processed-{state['item']}"]}

g = StateGraph(AggState)
g.add_node("start", start_agg)
g.add_node("process", process)
g.add_conditional_edges("start", fan_out, path_map=["process"])
g.add_edge(START, "start")
g.add_edge("process", END)

app_agg = g.compile()
result = app_agg.invoke({"items": ["a", "b", "c", "d", "e"]})
print(f"Input count: 5, Output count: {len(result['outputs'])}")
print(f"Outputs: {result['outputs']}")

## What You Learned

1. `Send` objects from conditional edges create **dynamic parallel workers** at runtime
2. The number of workers is determined by data, not graph structure
3. Each `Send` receives its own **isolated state**
4. Use `Annotated[list, operator.add]` to **aggregate** results from all workers
5. The **orchestrator-worker pattern** combines planning, parallel execution, and synthesis